<div style="display:flex; align-items:flex-start; gap:2rem; padding:1rem 0;">
  <div style="flex-shrink:0;">
    <img src="https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"
         style="height:36px; display:block; margin-bottom:0.75rem;"/>
    <img src="https://i.imgur.com/ITL8dZE.jpeg"
         style="width:260px; border-radius:4px; box-shadow:0 2px 8px rgba(0,0,0,0.15); display:block;"/>
  </div>
  <div style="flex:1; font-family:sans-serif;">
    <h1 style="font-size:1.6rem; font-weight:600; margin:0 0 0.25rem 0;">AI, ML and GenAI in the Lakehouse</h1>
    <p style="margin:0 0 0.75rem 0; color:#555; font-size:0.95rem;">
      <strong>Chapter 07-01 &mdash; Machine Learning at Scale with the NYC Taxi Dataset</strong><br/>
      Author: Bennie Haelen &nbsp;&bull;&nbsp; Date: 10-28-2024
    </p>
    <p style="margin:0 0 0.75rem 0; color:#333; font-size:0.9rem;">
      Demonstrates machine learning at scale using Apache Spark MLlib and Delta Lake
      on the full NYC Yellow Taxi 2023 dataset. Uses RandomForestRegressor for
      distributed parallel training, Optuna for hyperparameter tuning, and MLflow
      with Unity Catalog for experiment tracking and model registration.
    </p>
    <table style="border-collapse:collapse; font-size:0.85rem; color:#333; width:100%;">
      <tr style="background:#f5f5f5;">
        <td style="padding:4px 10px; font-weight:600;">Section</td>
        <td style="padding:4px 10px; font-weight:600;">Steps</td>
      </tr>
      <tr><td style="padding:4px 10px;"><strong>0 &mdash; Setup</strong></td>
          <td style="padding:4px 10px;">Constants &bull; imports &bull; MLflow registry URI</td></tr>
      <tr style="background:#f9f9f9;"><td style="padding:4px 10px;"><strong>1 &mdash; Data ingestion</strong></td>
          <td style="padding:4px 10px;">Load 12 monthly Parquet files &bull; estimate row count &bull; type conversions</td></tr>
      <tr><td style="padding:4px 10px;"><strong>2 &mdash; Preprocessing</strong></td>
          <td style="padding:4px 10px;">Trip duration &bull; filter outliers &bull; drop nulls</td></tr>
      <tr style="background:#f9f9f9;"><td style="padding:4px 10px;"><strong>3 &mdash; Feature engineering</strong></td>
          <td style="padding:4px 10px;">Temporal features &bull; numerical imputation &bull; categorical fill</td></tr>
      <tr><td style="padding:4px 10px;"><strong>4 &mdash; Vectorization</strong></td>
          <td style="padding:4px 10px;">StringIndexer &bull; OneHotEncoder &bull; VectorAssembler &bull; finalise dataset</td></tr>
      <tr style="background:#f9f9f9;"><td style="padding:4px 10px;"><strong>5 &mdash; Delta Lake storage</strong></td>
          <td style="padding:4px 10px;">Write to Delta &bull; OPTIMIZE</td></tr>
      <tr><td style="padding:4px 10px;"><strong>6 &mdash; Model training</strong></td>
          <td style="padding:4px 10px;">Train/test split &bull; Random Forest baseline &bull; evaluate RMSE</td></tr>
      <tr style="background:#f9f9f9;"><td style="padding:4px 10px;"><strong>7 &mdash; MLlib cross-validation</strong></td>
          <td style="padding:4px 10px;">ParamGridBuilder &bull; CrossValidator &bull; optimised RMSE</td></tr>
      <tr><td style="padding:4px 10px;"><strong>8 &mdash; Optuna tuning</strong></td>
          <td style="padding:4px 10px;">Objective function &bull; study &bull; MLflow tracking &bull; best params</td></tr>
      <tr style="background:#f9f9f9;"><td style="padding:4px 10px;"><strong>9 &mdash; Register in Unity Catalog</strong></td>
          <td style="padding:4px 10px;">Log best model &bull; register &bull; set champion alias &bull; verify</td></tr>
    </table>
  </div>
</div>

# Section 0: Setup and Prerequisites

###0-1: Libraries

In [0]:
%pip install optuna --quiet
dbutils.library.restartPython()

## 0-2: Constants

In [0]:
%run "../common/Constants"

In [0]:
# Inline fallback -- update these paths if the %run cell above is commented out.
# NYC_TAXI_DATASET_PATH  = "/mnt/nyc_taxi/raw"
# DELTA_NYC_DATASET_PATH = "/mnt/nyc_taxi/delta"
# CATALOG_NAME           = "book_ai_ml_lakehouse"
# ML_SCHEMA              = "ml_models"

## 0-3: Imports

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import (
    col, hour, dayofweek, date_format, unix_timestamp
)

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

import mlflow
import mlflow.spark

## 0-4: Configure MLflow for Unity Catalog

In [0]:
# All model registrations in this notebook target Unity Catalog.
mlflow.set_registry_uri("databricks-uc")

# Section 1: Data Ingestion and Initial Exploration

## 1-1: Load the dataset

In [0]:
files = [
    f"dbfs:{NYC_TAXI_DATASET_PATH}/yellow_tripdata_2023_{i:02}.parquet"
    for i in range(1, 13)
]

dfs = []
for file in files:
    df = spark.read.parquet(file)
    df = df.withColumn("passenger_count", df["passenger_count"].cast("double"))
    dfs.append(df)

taxi_df = dfs[0]
for df in dfs[1:]:
    taxi_df = taxi_df.union(df)

taxi_df.printSchema()
display(taxi_df)

## 1-2: Estimate row count by sampling

In [0]:
sample_fraction = 0.1
sample_count    = taxi_df.sample(fraction=sample_fraction, seed=42).count()
estimated_total = int(sample_count / sample_fraction)
print(f"Sampled rows:    {sample_count:,}")
print(f"Estimated total: {estimated_total:,}")

## 1-3: Type conversions

In [0]:
taxi_df = (
    taxi_df
    .withColumn("VendorID",              col("VendorID").cast("integer"))
    .withColumn("tpep_pickup_datetime",  col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("passenger_count",       col("passenger_count").cast("integer"))
    .withColumn("trip_distance",         col("trip_distance").cast("float"))
    .withColumn("RatecodeID",            col("RatecodeID").cast("integer"))
    .withColumn("PULocationID",          col("PULocationID").cast("integer"))
    .withColumn("DOLocationID",          col("DOLocationID").cast("integer"))
    .withColumn("payment_type",          col("payment_type").cast("integer"))
    .withColumn("fare_amount",           col("fare_amount").cast("float"))
    .withColumn("extra",                 col("extra").cast("float"))
    .withColumn("mta_tax",               col("mta_tax").cast("float"))
    .withColumn("tip_amount",            col("tip_amount").cast("float"))
    .withColumn("tolls_amount",          col("tolls_amount").cast("float"))
    .withColumn("improvement_surcharge", col("improvement_surcharge").cast("float"))
    .withColumn("total_amount",          col("total_amount").cast("float"))
    .withColumn("congestion_surcharge",  col("congestion_surcharge").cast("float"))
    .withColumn("airport_fee",           col("airport_fee").cast("float"))
)
taxi_df.printSchema()
display(taxi_df)

# Section 2: Data Preprocessing at Scale

## 2-1: Add a trip_duration column

In [0]:
taxi_df = taxi_df.withColumn(
    "trip_duration",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)

## 2-2: Filter out trips with unrealistic durations or distances

In [0]:
taxi_df = taxi_df.filter((col("trip_duration") > 0) & (col("trip_duration") <= 240))
taxi_df = taxi_df.filter((col("trip_distance") > 0) & (col("trip_distance") <= 100))
print(f"Rows after filtering: {taxi_df.count():,}")

## 2-3: Drop rows with missing critical values

In [0]:
taxi_df = taxi_df.dropna(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "trip_duration",
])
print(f"Rows after null drop: {taxi_df.count():,}")

# Section 3: Feature Engineering

## 3-1: Extract temporal features

In [0]:
taxi_df = taxi_df.withColumn("pickup_hour",  hour(col("tpep_pickup_datetime")))
taxi_df = taxi_df.withColumn("pickup_day",   dayofweek(col("tpep_pickup_datetime")))
taxi_df = taxi_df.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "M").cast(IntegerType())
)
taxi_df = taxi_df.withColumn(
    "pickup_year",
    date_format(col("tpep_pickup_datetime"), "yyyy").cast(IntegerType())
)

## 3-2: Impute missing values in numerical features

In [0]:
numerical_cols = ["trip_distance", "passenger_count", "pickup_hour"]

imputer = Imputer(
    inputCols=numerical_cols,
    outputCols=numerical_cols
).setStrategy("mean")

taxi_df = imputer.fit(taxi_df).transform(taxi_df)

## 3-3: Fill missing values in categorical features

In [0]:
categorical_columns = ["PULocationID", "DOLocationID", "pickup_day"]
taxi_df = taxi_df.fillna({col_name: -1 for col_name in categorical_columns})

# Section 4: Feature Transformation and Vectorization

## 4-1: Set up StringIndexers and OneHotEncoders

In [0]:
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_indexed", handleInvalid="keep")
    for c in categorical_columns
]
encoders = [
    OneHotEncoder(inputCol=c + "_indexed", outputCol=c + "_encoded", handleInvalid="keep")
    for c in categorical_columns
]
print(f"Indexers: {len(indexers)}  |  Encoders: {len(encoders)}")

## 4-2: Set up the VectorAssembler

In [0]:
assembler_input_cols = (
    [c + "_encoded" for c in categorical_columns] + numerical_cols
)
assembler = VectorAssembler(
    inputCols=assembler_input_cols,
    outputCol="features",
    handleInvalid="skip"
)

## 4-3: Execute the indexers

In [0]:
df_indexed = taxi_df
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)
df_indexed.printSchema()

## 4-4: Execute the encoders

In [0]:
df_encoded = df_indexed
for encoder in encoders:
    df_encoded = encoder.fit(df_encoded).transform(df_encoded)
df_encoded.printSchema()

## 4-5: Assemble the features column

In [0]:
df_assembled = assembler.transform(df_encoded)
df_assembled.printSchema()

## 4-6: Finalise the dataset
Select only `features` and `trip_duration`, renaming the latter to `label`
as required by Spark MLlib estimators.

In [0]:
df_final = (
    df_assembled
    .select("features", "trip_duration")
    .withColumnRenamed("trip_duration", "label")
)
print(df_final.columns)
display(df_final.limit(5))

# Section 5: Efficient Storage with Delta Lake

## 5-1: Write to Delta Lake

In [0]:
delta_path = f"dbfs:{DELTA_NYC_DATASET_PATH}"
print(f"Writing to: {delta_path}")
dbutils.fs.rm(delta_path, recurse=True)
df_final.write.format("delta").mode("overwrite").save(delta_path)
print("Write complete.")

## 5-2: Optimize the Delta table

In [0]:
spark.sql(f"OPTIMIZE delta.`{DELTA_NYC_DATASET_PATH}`")
print("OPTIMIZE complete.")

# Section 6: Model Training at Scale

## 6-1: Read back from Delta and split the data
Reading from the Delta table rather than reusing the in-memory DataFrame
ensures the model trains on exactly the data that was committed to storage,
and makes the training step independently restartable.

In [0]:
# Sample 20% of the data for training. This reduces training time from
# 90+ minutes to approximately 5 to 10 minutes while still demonstrating
# the full distributed training workflow on tens of millions of rows.
# Remove the sample() call to train on the complete dataset.
df_model = spark.read.format("delta").load(delta_path).sample(fraction=0.2, seed=42)

train_data, test_data = df_model.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train_data.count():,}")
print(f"Test rows:     {test_data.count():,}")

## 6-2: Train a Random Forest regressor
RandomForestRegressor is used instead of GBTRegressor because Random Forest
builds all trees in parallel across the cluster, whereas GBT builds trees
sequentially. On the full NYC Taxi dataset, Random Forest completes in
5 to 15 minutes compared to several hours for GBT with the same number of trees.

Random Forest handles non-linear relationships well, natively produces feature
importance scores, and is robust to outliers and noisy entries -- all of which
make it well-suited to the characteristics of this dataset.

We cache the training data before calling fit() to avoid re-reading from storage
on each tree. Caching reduces training time by 30 to 50 percent on datasets
of this size.

In [0]:
from mlflow.models.signature import infer_signature
import numpy as np

train_data.cache()
train_data.count()  # force the cache to materialise

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=8,
    seed=42
)

with mlflow.start_run(run_name="rf_baseline"):
    rf_model = rf.fit(train_data)

    mlflow.log_param("numTrees", rf.getNumTrees())
    mlflow.log_param("maxDepth", rf.getMaxDepth())

    mlflow.spark.log_model(
        rf_model,
        artifact_path="model",
        dfs_tmpdir=f"/Volumes/{CATALOG_NAME}/{ML_SCHEMA}/tmp"
    )
    print("Training complete.")

## 6-3: Evaluate the baseline model

In [0]:
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

predictions = rf_model.transform(test_data)
rmse = evaluator.evaluate(predictions)
print(f"Baseline Random Forest RMSE: {rmse:.2f} minutes")

# Section 7: Hyperparameter Tuning with MLlib CrossValidator
MLlib's CrossValidator and ParamGridBuilder perform an exhaustive grid search
with k-fold cross-validation, running entirely within the Spark execution engine.
Total fits = len(numTrees) x len(maxDepth) x numFolds = 2 x 3 x 3 = 18 model fits.

In [0]:
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

# TrainValidationSplit uses a single 80/20 split instead of k folds.
# Much lighter on memory than CrossValidator for large datasets.
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10])
    .build()
)

tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

with mlflow.start_run(run_name="rf_trainvalsplit"):
    tvs_model = tvs.fit(train_data)

    tvs_predictions = tvs_model.transform(test_data)
    tvs_rmse = evaluator.evaluate(tvs_predictions)

    best_rf = tvs_model.bestModel
    mlflow.log_param("best_numTrees", best_rf.getNumTrees)
    mlflow.log_param("best_maxDepth", best_rf.getMaxDepth())
    mlflow.log_metric("rmse", tvs_rmse)

    mlflow.spark.log_model(
        best_rf,
        artifact_path="model",
        dfs_tmpdir=f"/Volumes/{CATALOG_NAME}/{ML_SCHEMA}/tmp"
    )

print(f"TrainValidationSplit best RMSE: {tvs_rmse:.2f} minutes")
print(f"Best numTrees: {best_rf.getNumTrees}")
print(f"Best maxDepth: {best_rf.getMaxDepth()}")

# Section 8: Distributed Hyperparameter Tuning with Optuna
Optuna is the recommended hyperparameter optimization library on Databricks
Runtime 16.4 ML and above. It uses Tree of Parzen Estimators (TPE) by default,
which learns from previous trials to focus the search on promising regions of
the hyperparameter space rather than searching exhaustively like CrossValidator.

Each trial is logged as a nested MLflow child run under a parent experiment run,
making the full search history inspectable in the MLflow UI.

Note: Hyperopt with SparkTrials was the previous recommended approach but is no
longer included in Databricks Runtime ML after version 16.4 LTS ML. Use Optuna
for all new projects on current runtimes.

In [0]:
import optuna
import mlflow
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Suppress Optuna console output -- MLflow captures all trial data.
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 8-1: Define the objective function

In [0]:
def objective(trial):
    num_trees = trial.suggest_int("numTrees", 10, 30, step=10)
    max_depth = trial.suggest_int("maxDepth", 3, 5)       # reduced from 20

    with mlflow.start_run(nested=True):
        mlflow.log_params({"numTrees": num_trees, "maxDepth": max_depth})

        rf_trial = RandomForestRegressor(
            featuresCol="features",
            labelCol="label",
            numTrees=num_trees,
            maxDepth=max_depth,
            seed=42
        )

        model = rf_trial.fit(train_data)
        preds = model.transform(test_data)

        evaluator = RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="rmse"
        )
        rmse = evaluator.evaluate(preds)
        mlflow.log_metric("rmse", rmse)

    return rmse

## 8-2: Run the Optuna study

In [0]:
# TPESampler with a fixed seed makes results reproducible.
# n_jobs=1 is required when using Spark MLlib in the objective function
# since PySpark ML estimators are not thread-safe. Parallelism comes
# from Spark distributing each fit() call across the cluster.
study = optuna.create_study(
    direction="minimize",
    study_name="rf_nyc_taxi_tuning",
    sampler=optuna.samplers.TPESampler(seed=42)
)

with mlflow.start_run(run_name="optuna_rf_tuning"):
    study.optimize(
        objective,
        n_trials=5,
        n_jobs=1,
        show_progress_bar=True
    )

print(f"Best trial:       {study.best_trial.number}")
print(f"Best RMSE:        {study.best_trial.value:.4f}")
print(f"Best hyperparams: {study.best_trial.params}")

## 8-3: Inspect the results

In [0]:
import pandas as pd

trials_df = study.trials_dataframe()
trials_df = trials_df[["number", "value", "params_numTrees", "params_maxDepth", "state"]]
trials_df.columns = ["trial", "rmse", "numTrees", "maxDepth", "state"]
trials_df = trials_df.sort_values("rmse").reset_index(drop=True)

print("Top 5 trials by RMSE:")
print(trials_df.head(5).to_string(index=False))

## 8-4: Visualise the search (optional)

In [0]:
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_param_importances,
    )
    plot_optimization_history(study).show()
    plot_param_importances(study).show()
except ImportError:
    print("Install plotly for Optuna visualisations: %pip install plotly")

## 8-5: Extract best hyperparameters for Section 9
Store results in `best_hyperparams` so Section 9 can train the final model
without any additional changes.

In [0]:
best_hyperparams = study.best_trial.params
print(f"numTrees: {best_hyperparams['numTrees']}")
print(f"maxDepth: {best_hyperparams['maxDepth']}")

# Section 9: Log and Register the Best Model in Unity Catalog
With the best hyperparameters identified by Optuna in Section 8, we train a
final model and register it in Unity Catalog. The model alias 'champion'
replaces the deprecated stage-transition API.

## 9-1: Define the Unity Catalog model name

In [0]:
model_name       = "nyc_taxi_trip_duration_model"
unity_model_name = f"{CATALOG_NAME}.{ML_SCHEMA}.{model_name}"
print(f"Registering as: {unity_model_name}")

## 9-2: Train the best model and register in Unity Catalog

In [0]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# Define the signature manually since SparseVector
# cannot be auto-inferred from a pandas DataFrame.
input_schema  = Schema([ColSpec("double", "features")])
output_schema = Schema([ColSpec("double", "prediction")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="rf_best_model") as run:
    mlflow.log_param("numTrees", best_num_trees)
    mlflow.log_param("maxDepth", best_max_depth)

    rf_best_model = rf_best.fit(train_data)
    best_preds    = rf_best_model.transform(test_data)
    best_rmse     = evaluator.evaluate(best_preds)

    mlflow.log_metric("rmse", best_rmse)
    print(f"Best model RMSE: {best_rmse:.2f} minutes")

    mlflow.spark.log_model(
        rf_best_model,
        artifact_path="model",
        registered_model_name=unity_model_name,
        dfs_tmpdir=f"/Volumes/{CATALOG_NAME}/{ML_SCHEMA}/tmp",
        signature=signature
    )

print(f"Model registered: {unity_model_name}")

## 9-3: Assign the 'champion' alias
Model stage transitions (`transition_model_version_stage`) are deprecated in
MLflow 2.x and Unity Catalog. The recommended replacement is model aliases.

In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# search_model_versions is more reliable than get_registered_model().latest_versions
versions = client.search_model_versions(f"name='{unity_model_name}'")

if not versions:
    raise ValueError(f"No versions found for {unity_model_name}.")

latest_version = max(int(v.version) for v in versions)

client.set_registered_model_alias(
    name=unity_model_name,
    alias="champion",
    version=latest_version
)

print(f"Alias 'champion' set on version {latest_version} of {unity_model_name}")

## 9-4: Load and verify the registered model

In [0]:
loaded_model = mlflow.spark.load_model(
    f"models:/{unity_model_name}@champion",
    dfs_tmpdir=f"/Volumes/{CATALOG_NAME}/{ML_SCHEMA}/tmp"
)

sample       = test_data.limit(10)
sample_preds = loaded_model.transform(sample)
display(sample_preds.select("label", "prediction"))

# End of Notebook